# Self-Attention Master Guide: Mechanics & Training in Pure NumPy

Welcome! This single master notebook covers everything about **Self-Attention** from first principles using **100% pure NumPy**.

---

### Notebook Structure:
- **Part 1: Self-Attention Mechanics (Forward Step-by-Step)**
  1. Input Sequence Matrix ($X$)
  2. Shared Linear Projection Weight Matrices ($W_Q, W_K, W_V$)
  3. Raw Attention Similarity Scores ($S = Q \cdot K^T$)
  4. Scaling ($\frac{S}{\sqrt{d_k}}$) & Softmax Attention Weights ($A$)
  5. Weighted Sum of Values ($H = A \cdot V$)

- **Part 2: Training the Attention Network (Pure NumPy Backprop & SGD)**
  6. Task & Mini-Dataset Setup ("river bank overflowed" vs "money bank account")
  7. Forward Pass & Loss Function
  8. Analytical Backward Pass (Matrix Gradients for $W_Q, W_K, W_V$)
  9. Training Loop (SGD)
  10. Before vs. After Inspection (Seeing Attention Weights Adapt)

# PART 1: Self-Attention Mechanics (Forward Pass)

Let's start by understanding how a single forward pass of Self-Attention works.

In [28]:
import numpy as np

np.random.seed(42)
print(f"NumPy Version: {np.__version__}")

NumPy Version: 2.2.6


### Step 1: Input Sequence Matrix ($X$)

Sentence: **"The bank river overflowed"**
- Sequence length ($N$) = 4 tokens (`['The', 'bank', 'river', 'overflowed']`)
- Embedding dimension ($d_{\text{emb}}$) = 8

In [29]:
tokens = ['The', 'bank', 'river', 'overflowed']
seq_len = len(tokens)
emb_dim = 8

# Random input sequence matrix X (shape: 4 tokens x 8 dimensions)
X = np.random.randn(seq_len, emb_dim)
print(f"Sequence Matrix X shape: {X.shape} (4 tokens, 8 embedding dims)")

Sequence Matrix X shape: (4, 8) (4 tokens, 8 embedding dims)


### Step 2: Linear Projection Weights ($W_Q, W_K, W_V$) & Computing $Q, K, V$

- $W_Q, W_K, W_V$ shape: `(emb_dim, head_dim)` $\rightarrow$ `(8, 4)`.
- **Note:** These projection weights are **SHARED** across all tokens in the sequence!

In [30]:
head_dim = 4  # Dimension d_k for Q, K, V vectors

# Projection Weight Matrices
W_Q = np.random.randn(emb_dim, head_dim) * 0.1
W_K = np.random.randn(emb_dim, head_dim) * 0.1
W_V = np.random.randn(emb_dim, head_dim) * 0.1

# Compute Query, Key, Value matrices: (4, 8) @ (8, 4) -> (4, 4)
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print(f"Q shape: {Q.shape} | K shape: {K.shape} | V shape: {V.shape}")

Q shape: (4, 4) | K shape: (4, 4) | V shape: (4, 4)


### Step 3: Raw Attention Similarity Scores ($S = Q \cdot K^T$)

- Multiply Query matrix $Q$ `(4, 4)` by transposed Key matrix $K^T$ `(4, 4)` $\rightarrow$ Raw Scores `(4, 4)`.

In [ ]:
raw_scores = Q @ K.T ##Why ktranspose when the dimesnions are the same.
print("Raw Scores Matrix S (shape: 4, 4):")
print(raw_scores.round(3))

Raw Scores Matrix S (shape: 4, 4):
[[-0.011 -0.079  0.017  0.016]
 [ 0.06   0.124 -0.187  0.113]
 [-0.091  0.08   0.115 -0.022]
 [ 0.128 -0.046 -0.217  0.026]]


### Step 4: Scaling ($\frac{S}{\sqrt{d_k}}$) & Softmax Attention Weights ($A$)

Dividing by $\sqrt{d_k} = \sqrt{4} = 2.0$ stabilizes gradient flow in Softmax during training.

In [32]:
def softmax(logits, axis=-1):
    e = np.exp(logits - np.max(logits, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

d_k = head_dim
scaled_scores = raw_scores / np.sqrt(d_k)
attention_weights = softmax(scaled_scores)

print("Attention Weights Matrix A (Softmax output, each row sums to 1.0):")
print(attention_weights.round(4))

Attention Weights Matrix A (Softmax output, each row sums to 1.0):
[[0.2504 0.242  0.2539 0.2537]
 [0.2536 0.2618 0.2242 0.2604]
 [0.2362 0.2573 0.2619 0.2445]
 [0.2696 0.2472 0.2269 0.2563]]


### Step 5: Weighted Sum of Values ($H = A \cdot V$)

In [33]:
H = attention_weights @ V
print(f"Contextual Matrix H shape: {H.shape}")
print(f"Contextual vector for 'bank' (row 1): {H[1].round(3)}")

Contextual Matrix H shape: (4, 4)
Contextual vector for 'bank' (row 1): [-0.031  0.15  -0.233 -0.047]


# PART 2: Training the Attention Network (Pure NumPy Backprop & SGD)

Now let's turn this forward pass into a **trainable neural network**!

### Task Setup:
- Sentence 1: **"river bank overflowed"** $\rightarrow$ Label 0 (**Nature**)
- Sentence 2: **"money bank account"** $\rightarrow$ Label 1 (**Finance**)

We want $W_Q, W_K, W_V$ to learn through **Backpropagation** to focus on context tokens (`'river'` vs `'money'`) to classify `'bank'` correctly!

In [34]:
# Vocabulary setup
vocab = {"river": 0, "bank": 1, "overflowed": 2, "money": 3, "account": 4}
vocab_size = len(vocab)
emb_dim = 8
head_dim = 4

# Fixed Lookup Embedding Table E (5 words, 8 dims)
E = np.random.randn(vocab_size, emb_dim) * 0.1

# Dataset
dataset = [
    ([vocab["river"], vocab["bank"], vocab["overflowed"]], 0),
    ([vocab["money"], vocab["bank"], vocab["account"]], 1)
]

# Initialize Trainable Weights
W_Q = np.random.randn(emb_dim, head_dim) * 0.1
W_K = np.random.randn(emb_dim, head_dim) * 0.1
W_V = np.random.randn(emb_dim, head_dim) * 0.1

# Classifier Layer W_c (4, 2) and bias b_c (1, 2)
W_c = np.random.randn(head_dim, 2) * 0.1
b_c = np.zeros((1, 2))

print("Model parameters initialized.")

Model parameters initialized.


### Step 7 & 8: Forward Pass & Backward Pass Functions

In [35]:
def forward(token_ids, W_Q, W_K, W_V, W_c, b_c):
    X = E[token_ids]  # (seq_len, emb_dim)
    Q = X @ W_Q       # (seq_len, head_dim)
    K = X @ W_K       # (seq_len, head_dim)
    V = X @ W_V       # (seq_len, head_dim)
    
    d_k = head_dim
    S = (Q @ K.T) / np.sqrt(d_k)
    A = softmax(S, axis=-1)
    H = A @ V
    
    # Select contextual representation of 'bank' (token index 1)
    h_bank = H[1:2, :]  # (1, head_dim)
    
    # Output Logits & Softmax Probabilities
    logits = h_bank @ W_c + b_c
    probs = softmax(logits, axis=-1)
    
    cache = (X, Q, K, V, S, A, H, h_bank, logits, probs)
    return probs, cache

def backward(probs, label, cache, W_Q, W_K, W_V, W_c):
    X, Q, K, V, S, A, H, h_bank, logits, probs = cache
    d_k = head_dim
    
    y_onehot = np.zeros((1, 2))
    y_onehot[0, label] = 1.0
    
    # 1. dL/d_logits
    d_logits = probs - y_onehot
    
    # 2. Classifier Gradients
    dW_c = h_bank.T @ d_logits
    db_c = d_logits
    
    # 3. dL/dH
    dH = np.zeros_like(H)
    dH[1:2, :] = d_logits @ W_c.T
    
    # 4. dL/dV and dL/dA
    dV = A.T @ dH
    dA = dH @ V.T
    
    # 5. dL/dS (Softmax Backward)
    sum_dA_A = np.sum(dA * A, axis=-1, keepdims=True)
    dS = (A * (dA - sum_dA_A)) / np.sqrt(d_k)
    
    # 6. dL/dQ and dL/dK
    dQ = dS @ K
    dK = dS.T @ Q
    
    # 7. Final Projection Gradients dL/dW_Q, dL/dW_K, dL/dW_V
    dW_Q = X.T @ dQ
    dW_K = X.T @ dK
    dW_V = X.T @ dV
    
    return dW_Q, dW_K, dW_V, dW_c, db_c

### Step 9: Training Loop in Pure NumPy (SGD)

In [36]:
learning_rate = 0.1
epochs = 200

print("Training Self-Attention with Pure NumPy Backpropagation...\n")

for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for token_ids, label in dataset:
        # 1. Forward Pass
        probs, cache = forward(token_ids, W_Q, W_K, W_V, W_c, b_c)
        
        # 2. Cross-Entropy Loss
        loss = -np.log(probs[0, label] + 1e-12)
        total_loss += loss
        
        # 3. Backward Pass
        dW_Q, dW_K, dW_V, dW_c, db_c = backward(probs, label, cache, W_Q, W_K, W_V, W_c)
        
        # 4. Weight Updates (SGD)
        W_Q -= learning_rate * dW_Q
        W_K -= learning_rate * dW_K
        W_V -= learning_rate * dW_V
        W_c -= learning_rate * dW_c
        b_c -= learning_rate * db_c
        
    if epoch % 40 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | Total Loss: {total_loss:.4f}")

Training Self-Attention with Pure NumPy Backpropagation...

Epoch   1 | Total Loss: 1.4431
Epoch  40 | Total Loss: 1.4346
Epoch  80 | Total Loss: 1.4203
Epoch 120 | Total Loss: 1.3821
Epoch 160 | Total Loss: 1.2784
Epoch 200 | Total Loss: 1.0458


### Step 10: Inspect Learned Attention Weights!

Let's observe how the attention weights for `'bank'` adapted after training!

In [37]:
print("\n================ FINAL RESULTS ================\n")

# Sentence 1: "river bank overflowed"
probs1, cache1 = forward(dataset[0][0], W_Q, W_K, W_V, W_c, b_c)
attn1 = cache1[5]
pred1 = np.argmax(probs1)

print("Sentence 1: ['river', 'bank', 'overflowed']")
print(f"Predicted Class: {pred1} (0 = Nature/Water)")
print("Attention Weights for 'bank' (Row 1 -> ['river', 'bank', 'overflowed']):")
print(attn1[1].round(4))
print("-" * 55)

# Sentence 2: "money bank account"
probs2, cache2 = forward(dataset[1][0], W_Q, W_K, W_V, W_c, b_c)
attn2 = cache2[5]
pred2 = np.argmax(probs2)

print("Sentence 2: ['money', 'bank', 'account']")
print(f"Predicted Class: {pred2} (1 = Finance)")
print("Attention Weights for 'bank' (Row 1 -> ['money', 'bank', 'account']):")
print(attn2[1].round(4))


================ FINAL RESULTS ================

Sentence 1: ['river', 'bank', 'overflowed']
Predicted Class: 0 (0 = Nature/Water)
Attention Weights for 'bank' (Row 1 -> ['river', 'bank', 'overflowed']):
[0.3332 0.3334 0.3334]
-------------------------------------------------------
Sentence 2: ['money', 'bank', 'account']
Predicted Class: 1 (1 = Finance)
Attention Weights for 'bank' (Row 1 -> ['money', 'bank', 'account']):
[0.3333 0.3331 0.3336]
